In [1]:
import pandas as pd
import numpy as np
import pickle

In [5]:
dataset_paths = [
    '/mnt/czi-sci-ai/project-rbio/AutoSync/Datasets/PertQA-DE/hepg2-train-v0.2.0-no-augmentation.csv',
    '/mnt/czi-sci-ai/project-rbio/AutoSync/Datasets/PertQA-DE/hepg2-test-v0.2.0-no-augmentation.csv',
    '/mnt/czi-sci-ai/project-rbio/AutoSync/Datasets/PertQA-DE/rpe1-train-v0.2.0-no-augmentation.csv',
    '/mnt/czi-sci-ai/project-rbio/AutoSync/Datasets/PertQA-DE/rpe1-test-v0.2.0-no-augmentation.csv',
    '/mnt/czi-sci-ai/project-rbio/AutoSync/Datasets/PertQA-DE/jurkat-train-v0.2.0-no-augmentation.csv',
    '/mnt/czi-sci-ai/project-rbio/AutoSync/Datasets/PertQA-DE/jurkat-test-v0.2.0-no-augmentation.csv',
    '/mnt/czi-sci-ai/project-rbio/AutoSync/Datasets/PertQA-DE/k562-train-v0.2.0-no-augmentation.csv',
    '/mnt/czi-sci-ai/project-rbio/AutoSync/Datasets/PertQA-DE/k562-test-v0.2.0-no-augmentation.csv'
]

In [6]:
embedding_file = '/mnt/czi-sci-ai/project-rbio/repr/gene2vec_embeddings.pkl'
save_file = '/mnt/czi-sci-ai/project-rbio/repr/gene2vec_embeddings_filled.pkl'

In [7]:
with open(embedding_file, "rb") as f:
    emb_dict = pickle.load(f)
        
dfs = [pd.read_csv(path) for path in dataset_paths]
df_training = pd.concat(dfs, ignore_index=True)

genes = pd.unique(df_training[["gene_perturbed", "gene_monitored"]].values.ravel())
all_genes = sorted(set(genes))
gene_to_index = {gene: i for i, gene in enumerate(all_genes)}

name_to_embedding = {}
missing = 0
for gene, idx in gene_to_index.items():
    try:
        name_to_embedding[gene.lower()] = np.asarray(
            emb_dict[gene.lower()], dtype=np.float32
        )
    except KeyError:
        missing += 1
        print(f"WARNING: Missing embedding for gene {gene} (#{missing})")
        rand_emb = np.random.randn(len(next(iter(emb_dict.values())))).astype(
            np.float32
        )
        name_to_embedding[gene.lower()] = rand_emb


In [9]:
print(f'saving in {save_file}')

saving in /mnt/czi-sci-ai/project-rbio/repr/gene2vec_embeddings_filled.pkl


In [10]:
with open(save_file, 'wb') as f:
    pickle.dump(name_to_embedding, f)